# 1. Mount google drive

In [23]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
# Set working directory

import os, sys, shutil
repo_root = "/content/drive/Othercomputers/My Laptop/SmartHVAC-Studio"
colab_dir = os.path.join(repo_root, "colab")

# Ensure eplus utility is copied for imports
eplus_src = os.path.join(repo_root, "EnergyPlus utility")
eplus_dest = os.path.join(colab_dir, "eplus")
if not os.path.exists(eplus_dest):
    shutil.copytree(eplus_src, eplus_dest)

os.chdir(colab_dir)
sys.path.append(colab_dir)
print(f"Working directory set to: {os.getcwd()}")


Working directory set to: /content/drive/Othercomputers/My Laptop/SmartHVAC-Studio/colab


# 2. Ollama local run

In [25]:
# 1. Install missing dependencies & GPU utils
!sudo apt-get install -y zstd
!sudo apt update && sudo apt install pciutils lshw -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 58 not upgraded.
Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:5 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
58 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of co

In [26]:
# 2. Install Ollama Linux Engine
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [27]:
# 3. Install Python SDK
!pip install ollama

In [ ]:
# 1. Kill any existing instances
!pkill ollama

In [28]:
# 4. Set host environment variables to allow connections
import os
import time
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'

In [29]:
# 2. Safely copy the massive brain from Drive to Colab's fast internal disk (~30 seconds)
print("Copying model from Drive to local disk...")
!mkdir -p /root/.ollama/models
!cp -r "/content/drive/Othercomputers/My Laptop/SmartHVAC-Studio/colab/ollama_models/"* /root/.ollama/models/
print("Copy complete!")

Copying model from Drive to local disk...
Copy complete!


In [30]:
# 3. Boot Server normally (it will use the fast internal disk, NOT Drive)
!nohup ollama serve > ollama.log 2>&1 &
time.sleep(5)
print("✅ Ollama Server Awake and ready!")

✅ Ollama Server Awake and ready!


In [31]:
# 6. Test if it is running
!curl http://localhost:11434

Ollama is running

In [ ]:
# 7. Pull the model!
!ollama pull gemma3:4b

In [ ]:
!ollama rm gemma3:270m

In [12]:
!ollama list

NAME         ID              SIZE      MODIFIED       
gemma3:4b    a2af6cc3eb7f    3.3 GB    15 seconds ago    


In [32]:
"""### Testing on Text Input"""

import ollama
response = ollama.chat(model='gemma3:4b', messages=[
  {'role': 'user', 'content': 'what is rgb, answer short'},
])
print(response['message']['content'])

RGB stands for Red, Green, and Blue. It’s a color model used in displays and digital images, where colors are created by mixing these three primary colors. 

Essentially, it’s how computers represent color! 



# SmartHVAC Studio: Backend Worker (Layer 3)
This executable notebook acts as the **Coordination Engine**. It polls Firebase for new jobs, runs AI generation (Layer 4), executes EnergyPlus simulations (Layer 5), and uploads results.

---

In [33]:
import os, sys, importlib, shutil

# 1. Clean up stale simulation folders
for folder in ['../sim_runs', 'sim_runs', 'RunFiles']:
    if os.path.exists(folder):
        shutil.rmtree(folder)

# 2. Force Python to reload the latest python files from disk
sys.path.append(os.getcwd())
import backend.geometry_util as geometry_util
import simulation_runner
importlib.reload(geometry_util)
importlib.reload(simulation_runner)

print("✅ Cache cleared and modules reloaded. Ready to simulate.")


✅ Cache cleared and modules reloaded. Ready to simulate.


## 2. Setup Environment

In [34]:
# 1. Install Backend Dependencies
!pip install -r requirements.txt

# 2. Install Advisor's EnergyPlus Utility (from GitHub)
# Note: This installs the specific dev branch required for EKF hooks
!pip install -q "energy-plus-utility @ git+https://github.com/mugalan/energy-plus-utility.git@dev"

print("Dependencies installed.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Dependencies installed.


## 3. Authentication

In [35]:
# Check for Service Account Key
key_path = "serviceAccountKey.json"

if not os.path.exists(key_path):
    print("❌ ERROR: serviceAccountKey.json not found.")
    print("Please upload your Firebase Service Account JSON key to the file browser on the left.")
else:
    print("✅ Found serviceAccountKey.json")

✅ Found serviceAccountKey.json


## 3. Initialize Modules

In [36]:
import firebase_admin
from firebase_admin import credentials, firestore, storage
import json
import time
import difflib # For Diff Viewer

In [37]:
from backend.firebase_connector import FirebaseConnector
from backend.ai_generator import AIPipelines
from simulation_runner import run_simulation_job
import time

# Initialize Connections
try:
    fb = FirebaseConnector(key_path)
    ai = AIPipelines() # API Key can be passed here or set in ENV
    print("Backend Modules Initialized Successfully.")
except Exception as e:
    print("Initialization Failed:", e)

[Firebase] Connected to project: smarthvac-studio-b84c4
Backend Modules Initialized Successfully.


## 4. Verify epw found and downloaded to runtime

In [38]:
import os

# Expected path relative to Colab root
file_path = "weather/USA_IL_Chicago-OHare.Intl.AP.725300_TMY3.epw"

if os.path.exists(file_path):
    print(f"✅ FOUND: {file_path}")
    print(f"   Size: {os.path.getsize(file_path)/1024:.1f} KB")
else:
    print(f"❌ MISSING: {file_path}")
    if os.path.exists("weather"):
        print(f"   Contents of 'weather/': {os.listdir('weather')}")
    else:
        print("   'weather' folder does not exist!")


✅ FOUND: weather/USA_IL_Chicago-OHare.Intl.AP.725300_TMY3.epw
   Size: 1610.1 KB


## 5. Verify Base.idf found and downloaded to runtime

In [39]:
import difflib

def generate_and_upload_diff(base_path, new_path, job_id):
    """Generates an HTML diff between Base.idf and in.idf and uploads it."""
    try:
        with open(base_path, 'r') as f:
            base_lines = f.readlines()
        with open(new_path, 'r') as f:
            new_lines = f.readlines()

        diff_html = difflib.HtmlDiff().make_file(
            base_lines, new_lines,
            fromdesc='Base Template',
            todesc='Generated IDF',
            context=True,
            numlines=3
        )

        diff_path = f"jobs/{job_id}/diff.html"
        with open(diff_path, "w") as f:
            f.write(diff_html)

        bucket = storage.bucket()
        blob = bucket.blob(f"jobs/{job_id}/diff.html")
        blob.upload_from_filename(diff_path, content_type="text/html")
        print(f"   📊 Uploaded Diff HTML to Storage.")
        return True
    except Exception as e:
        print(f"   ❌ Failed to generate diff: {e}")
        return False

# Verify Base.idf exists locally (from Drive, NOT from Firebase Storage)
base_path = 'templates/Base.idf'
if os.path.exists(base_path):
    print(f"✅ Base.idf found: {base_path} ({os.path.getsize(base_path)/1024:.1f} KB)")
else:
    print(f"❌ Base.idf NOT found at {base_path}!")


✅ Base.idf found: templates/Base.idf (14.0 KB)


## 6. Main polling loop

In [40]:
import time
from datetime import datetime
import traceback

def run_backend_loop(db):
    print("🚀 Backend is running... Waiting for jobs.")
    print("   - Watching 'jobs' collection for 'queued' tasks.")
    print("   - Watching 'test_connectivity' collection for 'test_connection' tasks.")

    while True:
        try:
            # ---------------------------------------------------------
            # 1. Check for CONNECTION TESTS (Fast Path)
            # ---------------------------------------------------------
            test_docs = db.collection("test_connectivity").where("status", "==", "test_connection").stream()

            for doc in test_docs:
                job_id = doc.id
                data = doc.to_dict()
                print(f"🔹 Found Connection Test: {job_id}")

                # Get Toggles (Default to True if missing)
                do_gemini = data.get('checkGemini', True)
                do_openai = data.get('checkOpenAI', True)
                do_hf = data.get('checkHF', True)

                # Perform checks using your existing AI class
                # Pass the flags to the method
                results = ai.test_connections(check_openai=do_openai, check_gemini=do_gemini, check_hf=do_hf)

                # Update Firestore
                db.collection("test_connectivity").document(job_id).update({
                    "status": "tested",
                    "testResults": results,
                    "processedAt": datetime.now()
                })
                print(f"   ✅ Test Completed. Updated {job_id}")

            # ---------------------------------------------------------
            # 2. Check for SIMULATION JOBS (Normal Path)
            # ---------------------------------------------------------
            job_docs = db.collection("jobs").where("status", "==", "queued").stream()

            for doc in job_docs:
                job_id = doc.id
                data = doc.to_dict()
                print(f"🔸 Found Queued Job: {job_id}")

                # Mark as running immediately
                db.collection("jobs").document(job_id).update({"status": "running"})

                try:
                    # Extract Inputs from new UI
                    nlp_input = data.get("nlpInputText", "")
                    sim_settings = data.get("simulationConfig", {})

                    # Log what we received
                    print(f"   Input: {nlp_input[:50]}...")
                    print(f"   Settings: {sim_settings}")


                    # --- PHASE 3: THE SYSTEM LOOP ---
                    run_mode = data.get("runMode", "ai")

                    import os
                    idf_save_path = f"RunFiles/{job_id}_in.idf"
                    os.makedirs("RunFiles", exist_ok=True)

                    if run_mode == "minimal":
                        print("   [Phase 3] Safe Test Mode: Using Minimal.idf bypass...")
                        import shutil
                        # minimal_path = "../Example IDF files/Minimal.idf"
                        minimal_path = "../Example IDF files/Exercise1A.idf"

                        if not os.path.exists(minimal_path):
                            raise Exception(f"Could not find Minimal.idf at {minimal_path}")
                        shutil.copy(minimal_path, idf_save_path)
                    else:
                        selected_model = data.get("selectedModel", "openai")
                        print(f"   [Phase 3] Generating IDF using {selected_model}...")

                        final_idf_string = ai.generate_idf_from_text(nlp_input, sim_settings, model_type=selected_model)

                        if final_idf_string.startswith("! Error"):
                            raise Exception(f"AI Generation Failed: {final_idf_string}")

                        # Save IDF
                        with open(idf_save_path, "w", encoding="utf-8") as f:
                            f.write(final_idf_string)


                    # Epw path usually defaults to "weather.epw" or from sim_settings
                    epw_name = sim_settings.get("weather_file", "weather.epw")
                    if not os.path.exists(epw_name) and os.path.exists("weather/" + epw_name):
                         epw_name = "weather/" + epw_name
                    elif not os.path.exists(epw_name):
                         epw_name = "weather.epw"

                    sim_results = simulation_runner.run_simulation_job(
                        job_id=job_id,
                        idf_path=idf_save_path,
                        epw_path=epw_name,
                        config=sim_settings,
                        output_dir_base="sim_runs"
                    )
                    print("   [Phase 3] Simulation Complete! Uploading results to Storage...")

                    # Upload to Storage
                    bucket = storage.bucket()
                    result_urls = {}

                    for key, file_path in sim_results.items():
                        if os.path.exists(file_path):
                            blob_path = f"jobs/{job_id}/{os.path.basename(file_path)}"
                            blob = bucket.blob(blob_path)

                            # c_type = "image/png" if key == "plot" else "text/csv"
                            #c_type = "image/png" if key.startswith("plot") else "text/csv"

                            if key.startswith("plot"):
                                c_type = "image/png"
                            elif key == "idf":
                                c_type = "text/plain"
                            elif key == "summary":            # <--- ADD THIS LINE
                                c_type = "text/html"          # <--- ADD THIS LINE
                            else:
                                c_type = "text/csv"


                            blob.upload_from_filename(file_path, content_type=c_type)

                            # Store the direct URL
                            try:
                                blob.make_public()
                                result_urls[key] = blob.public_url
                            except:
                                result_urls[key] = f"gs://{bucket.name}/{blob_path}"

                    db.collection("jobs").document(job_id).update({
                        "status": "done",
                        "resultPaths": result_urls,
                        "completedAt": datetime.now()
                    })
                    print(f"   ✅ Job Finished: {job_id}\n")

                except Exception as e:
                    error_msg = str(e)
                    print(f"   ❌ Job Failed: {error_msg}")
                    traceback.print_exc()

                    db.collection("jobs").document(job_id).update({
                        "status": "error",
                        "errorMessage": error_msg
                    })

            # Sleep to prevent quota exhaustion
            time.sleep(3)

        except Exception as main_e:
            print(f"⚠️ Loop Error: {main_e}")
            time.sleep(5)

# ---------------------------------------------------------------------
# START THE LOOP (Runs indefinitely)
# Ensure 'fb' is defined (from previous cells)
# ---------------------------------------------------------------------
if __name__ == "__main__":
    if 'fb' in globals():
        # Access the internal firestore client from your connector
        if hasattr(fb, 'db'):
            db_client = fb.db
        else:
            # Fallback if fb is the client itself
            db_client = fb

        run_backend_loop(db_client)
    else:
        print("❌ Error: 'fb' variable not found. Please run the Firebase Initialization cell first.")


🚀 Backend is running... Waiting for jobs.
   - Watching 'jobs' collection for 'queued' tasks.
   - Watching 'test_connectivity' collection for 'test_connection' tasks.
🔸 Found Queued Job: job_20260429_185508
   Input: Simulate a building that is 30.00 meters long, 15....
   Settings: {'weather_file': 'USA_IL_Chicago-OHare.Intl.AP.725300_TMY3.epw', 'run_type': 'design_day'}
   [Phase 3] Generating IDF using ollama...
[AI] Generating Modular IDF using model: ollama
[AI RAG] Running Pass 1 (Planner) to extract keywords...
[AI RAG] Extracted Keywords: ['building', 'dimensions', 'window-to-wall ratio', 'occupancy', 'lighting', 'equipment', 'walls', 'insulation']
[AI RAG] Filtered Menu down to 15 items: ['Light Exterior Wall', 'Light Roof/Ceiling', 'Light Partitions', 'Light Floor', 'Light Furnishings', 'Medium Exterior Wall', 'Medium Roof/Ceiling', 'Medium Partitions', 'Medium Floor', 'Medium Furnishings', 'Heavy Exterior Wall', 'Heavy Roof/Ceiling', 'Heavy Partitions', 'Heavy Floor', 'Heav

/usr/local/lib/python3.12/dist-packages/google/cloud/firestore_v1/base_collection.py:317: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)


KeyboardInterrupt: 